# Train the compact CNN

This notebook trains the compact custom CNN (CCCNN) on the processed 64x64 patches. Run the preprocessing notebook first so that data_processed contains the class folders. The notebook creates a grouped train, validation, and test split, trains the model, saves the best validation checkpoint, and includes prediction cells for held-out dataset patches and a user image.

Patches with the same source fabric ID stay in one split. This gives a more realistic test of generalization to unseen fabric images.


In [1]:
from pathlib import Path
from collections import Counter
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "data_processed").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root with data_processed/ and notebooks/.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data_processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "ccc_nn_best.pt"
SPLIT_PATH = ARTIFACT_DIR / "split_manifest.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 64
MAX_EPOCHS = 25
PATIENCE = 5
LEARNING_RATE = 1e-3

print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Processed images:", DATA_DIR)


Project: E:\Ganesh Sarathi\College\Coding\DIP\Fabric Defect Detection\FabricDefectDetectionDIP
Device: cpu
Processed images: E:\Ganesh Sarathi\College\Coding\DIP\Fabric Defect Detection\FabricDefectDetectionDIP\data_processed


## Load paths and make a grouped split

Filenames in this dataset use the source-image ID before _patch. We use that ID as the group when splitting. The validation and test proportions are approximate because the split keeps each source image intact.


In [2]:
class_names = sorted(path.name for path in DATA_DIR.iterdir() if path.is_dir())
if not class_names:
    raise FileNotFoundError(f"No class folders found in {DATA_DIR}")

records = []
for label_id, class_name in enumerate(class_names):
    for path in sorted((DATA_DIR / class_name).glob("*.png")):
        source_id = path.stem.split("_patch", 1)[0]
        if source_id == path.stem:
            raise ValueError(f"Expected a source ID followed by _patch in filename: {path.name}")
        records.append({
            "path": path,
            "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
            "class_name": class_name,
            "label": label_id,
            "source_id": source_id,
        })

if not records:
    raise FileNotFoundError(f"No PNG patches found in {DATA_DIR}")

groups = np.array([r["source_id"] for r in records])
indices = np.arange(len(records))
trainval_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(indices, groups=groups))
train_idx_local, val_idx_local = next(
    GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=SEED + 1)
    .split(trainval_idx, groups=groups[trainval_idx])
)
train_idx = trainval_idx[train_idx_local]
val_idx = trainval_idx[val_idx_local]

for split_name, split_idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    present = {records[i]["class_name"] for i in split_idx}
    missing = set(class_names) - present
    if missing:
        raise ValueError(f"{split_name} split is missing classes: {sorted(missing)}. Rerun with another SEED.")

split_rows = []
for split_name, split_idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    for i in split_idx:
        split_rows.append({**records[i], "path": str(records[i]["path"]), "split": split_name})
manifest = pd.DataFrame(split_rows)
manifest.drop(columns=["path"]).to_csv(SPLIT_PATH, index=False)

summary = pd.crosstab(manifest["split"], manifest["class_name"])
summary["total"] = summary.sum(axis=1)
display(summary)
print("Grouped split manifest saved to:", SPLIT_PATH)


class_name,good,hole,objects,oil spot,thread error,total
split,,,,,,
test,3481,52,141,102,64,3840
train,16196,233,562,427,438,17856
validation,3493,52,134,107,118,3904


Grouped split manifest saved to: E:\Ganesh Sarathi\College\Coding\DIP\Fabric Defect Detection\FabricDefectDetectionDIP\artifacts\split_manifest.csv


## Dataset and augmentation

Training batches use weighted sampling to show defect classes more often. Training images receive random flips and small rotations; defect images also receive mild brightness and contrast variation. Validation and test images are not augmented.


In [3]:
class FabricPatchDataset(Dataset):
    def __init__(self, rows, training=False):
        self.rows = rows
        self.training = training
        self.to_tensor = transforms.ToTensor()
        self.color_jitter = transforms.ColorJitter(brightness=0.12, contrast=0.12)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
        if self.training:
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
            if random.random() < 0.5:
                image = ImageOps.flip(image)
            angle = random.uniform(-10, 10)
            image = image.rotate(angle, resample=Image.Resampling.BILINEAR)
            if row["class_name"] != "good" and random.random() < 0.5:
                image = self.color_jitter(image)
        return self.to_tensor(image), int(row["label"])

train_rows = [records[i] for i in train_idx]
val_rows = [records[i] for i in val_idx]
test_rows = [records[i] for i in test_idx]

train_counts = Counter(row["label"] for row in train_rows)
sample_weights = [1.0 / np.sqrt(train_counts[row["label"]]) for row in train_rows]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(FabricPatchDataset(train_rows, training=True), batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(FabricPatchDataset(val_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(FabricPatchDataset(test_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("Training samples:", len(train_rows), "| validation:", len(val_rows), "| test:", len(test_rows))


Training samples: 17856 | validation: 3904 | test: 3840
